# Project 9 Enhanced — Country Development Clustering with K-Means

## From university lab to decision-support analysis

This notebook **does not replace the original university lab**. It extends the same 167-country clustering exercise into a stronger, interview-ready project.

### Original lab preserved
The original notebook:
- loaded `Country-data.csv`;
- used the Yellowbrick K-Elbow Visualizer;
- applied K-Means with **K = 3**;
- assigned cluster labels to countries.

### Enhancements in this version
- explicit data-quality checks;
- feature scaling before K-Means;
- Yellowbrick K-Elbow on the scaled matrix;
- model selection with **silhouette, Calinski-Harabasz and Davies-Bouldin**;
- comparison of the original unscaled K=3 solution with the scaled solution;
- PCA visualisation;
- interpretable cluster profiles;
- aid-priority / development-need ranking;
- representative countries per cluster;
- stability testing across random seeds;
- reusable country-assignment function.

> Clusters are statistical development profiles, not moral rankings of countries.

In [ ]:
import sys, subprocess, importlib.util

if importlib.util.find_spec("yellowbrick") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "yellowbrick"])

print("Environment ready.")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    adjusted_rand_score
)
from yellowbrick.cluster import KElbowVisualizer

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

print("Libraries imported.")

## 1. Load the same 167-country HELP International dataset

The dataset contains nine socio-economic and health indicators:
- child mortality
- exports
- health spending
- imports
- income
- inflation
- life expectancy
- fertility
- GDP per capita

The country name is retained only as an identifier, not a clustering feature.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/392781/Country-Clustering-Analysis/master/country-data.csv"

data = pd.read_csv(DATA_URL)

print("Shape:", data.shape)
print("Columns:", data.columns.tolist())
display(data.head(10))

expected = {
    "country","child_mort","exports","health","imports",
    "income","inflation","life_expec","total_fer","gdpp"
}
assert expected.issubset(data.columns)
assert data["country"].nunique() == len(data)
print("Dataset checks passed.")

## 2. Data-quality audit

In [ ]:
audit = pd.DataFrame({
    "dtype": data.dtypes.astype(str),
    "missing": data.isna().sum(),
    "missing_pct": data.isna().mean(),
    "unique": data.nunique()
})
display(audit)

print("Duplicate countries:", data["country"].duplicated().sum())
print("Duplicate full rows:", data.duplicated().sum())
display(data.describe().T.round(2))

## 3. Exploratory analysis

The original lab worked directly with the raw numeric values. Here we first inspect how different their scales are.

In [ ]:
features = [
    "child_mort","exports","health","imports","income",
    "inflation","life_expec","total_fer","gdpp"
]

fig, axes = plt.subplots(3, 3, figsize=(15, 11))
for ax, col in zip(axes.ravel(), features):
    ax.hist(data[col], bins=25)
    ax.set_title(col)
    ax.set_xlabel("value")
    ax.set_ylabel("countries")
plt.tight_layout()
plt.show()

scale_table = data[features].agg(["min","median","max"]).T
display(scale_table.round(2))

In [ ]:
corr = data[features].corr()
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(corr, vmin=-1, vmax=1)
ax.set_xticks(range(len(features)), features, rotation=45, ha="right")
ax.set_yticks(range(len(features)), features)
ax.set_title("Correlation matrix of development indicators")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

## 4. Why scaling matters for K-Means

K-Means minimises Euclidean distances. A variable such as GDP per capita can have values in the tens of thousands while health spending is often single digits.

Without scaling, the larger-magnitude variables can dominate the distance calculation. We therefore standardise all nine indicators before fitting the enhanced model.

In [ ]:
X_raw = data[features].copy()

scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

scaled_check = pd.DataFrame(X, columns=features).agg(["mean","std"]).T
display(scaled_check.round(3))

print("Scaled matrix shape:", X.shape)
print("Finite:", np.isfinite(X).all())

## 5. Yellowbrick K-Elbow Visualizer — retained from the university method

The original lab used `KElbowVisualizer`. We keep that method, but apply it to the **scaled** feature matrix.

In [ ]:
# Keep the university Yellowbrick method when the installed versions are compatible.
# Yellowbrick 1.5 predates some newer scikit-learn releases, so a deterministic
# manual elbow fallback is included to keep the notebook reproducible.

fig, ax = plt.subplots(figsize=(8, 5))
elbow_model = KMeans(random_state=RANDOM_STATE, n_init=20)

try:
    visualizer = KElbowVisualizer(
        elbow_model,
        k=(2, 11),
        metric="distortion",
        timings=False,
        ax=ax,
        force_model=True
    )
    visualizer.fit(X)
    visualizer.finalize()
    elbow_k = visualizer.elbow_value_
    elbow_method = "Yellowbrick KElbowVisualizer"
except Exception as exc:
    print("Yellowbrick compatibility fallback:", type(exc).__name__)
    ks = np.arange(2, 11)
    inertias = []
    for k in ks:
        m = KMeans(
            n_clusters=int(k),
            init="k-means++",
            n_init=30,
            random_state=RANDOM_STATE
        ).fit(X)
        inertias.append(m.inertia_)

    inertias = np.asarray(inertias, dtype=float)

    # Select the point furthest from the straight line joining the first/last points.
    x = (ks - ks.min()) / (ks.max() - ks.min())
    y = (inertias - inertias.min()) / (inertias.max() - inertias.min())
    p1 = np.array([x[0], y[0]])
    p2 = np.array([x[-1], y[-1]])
    distances = []
    for xi, yi in zip(x, y):
        p = np.array([xi, yi])
        distances.append(
            abs(np.cross(p2-p1, p1-p)) / np.linalg.norm(p2-p1)
        )
    elbow_k = int(ks[int(np.argmax(distances))])
    elbow_method = "manual inertia-knee fallback"

    ax.plot(ks, inertias, marker="o")
    ax.axvline(elbow_k, linestyle="--", label=f"elbow K={elbow_k}")
    ax.set_xlabel("K")
    ax.set_ylabel("Inertia")
    ax.set_title("K-Means elbow curve")
    ax.set_xticks(ks)
    ax.legend()
    ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

print("Elbow method:", elbow_method)
print("Elbow K:", elbow_k)

## 6. Stronger K selection with multiple internal metrics

No single clustering metric is perfect. For K = 2 to 10 we compare:
- **Silhouette** — higher is better
- **Calinski-Harabasz** — higher is better
- **Davies-Bouldin** — lower is better
- inertia — lower is better but always decreases as K rises

The enhanced model selects the K with the highest silhouette score, while the original K=3 result remains available for comparison.

In [ ]:
selection_rows = []

for k in range(2, 11):
    model = KMeans(
        n_clusters=k,
        init="k-means++",
        n_init=30,
        random_state=RANDOM_STATE
    )
    labels = model.fit_predict(X)
    selection_rows.append({
        "k": k,
        "inertia": model.inertia_,
        "silhouette": silhouette_score(X, labels),
        "calinski_harabasz": calinski_harabasz_score(X, labels),
        "davies_bouldin": davies_bouldin_score(X, labels)
    })

selection = pd.DataFrame(selection_rows)
BEST_K = int(selection.loc[selection["silhouette"].idxmax(), "k"])

display(selection.round(4))
print("Best K by silhouette:", BEST_K)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(selection["k"], selection["silhouette"], marker="o", label="Silhouette")
ax.set_xlabel("K")
ax.set_ylabel("Silhouette score")
ax.set_title("K selection by silhouette")
ax.set_xticks(selection["k"])
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(selection["k"], selection["davies_bouldin"], marker="o")
ax.set_xlabel("K")
ax.set_ylabel("Davies-Bouldin score")
ax.set_title("K selection by Davies-Bouldin (lower is better)")
ax.set_xticks(selection["k"])
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 7. Compare the original university K=3 approach with the scaled K=3 solution

This demonstrates exactly why the enhancement is useful without deleting or rewriting the original exercise.

In [ ]:
original_k3 = KMeans(
    n_clusters=3,
    init="k-means++",
    n_init=30,
    random_state=10
)
original_labels = original_k3.fit_predict(X_raw)

scaled_k3 = KMeans(
    n_clusters=3,
    init="k-means++",
    n_init=30,
    random_state=10
)
scaled_k3_labels = scaled_k3.fit_predict(X)

comparison = pd.DataFrame([
    {
        "solution":"Original-style unscaled K=3",
        "silhouette":silhouette_score(X_raw, original_labels),
        "calinski_harabasz":calinski_harabasz_score(X_raw, original_labels),
        "davies_bouldin":davies_bouldin_score(X_raw, original_labels)
    },
    {
        "solution":"Enhanced scaled K=3",
        "silhouette":silhouette_score(X, scaled_k3_labels),
        "calinski_harabasz":calinski_harabasz_score(X, scaled_k3_labels),
        "davies_bouldin":davies_bouldin_score(X, scaled_k3_labels)
    }
]).set_index("solution")

display(comparison.round(4))

## 8. Fit the enhanced final K-Means model

In [ ]:
final_kmeans = KMeans(
    n_clusters=BEST_K,
    init="k-means++",
    n_init=50,
    random_state=RANDOM_STATE
)
cluster = final_kmeans.fit_predict(X)

results = data.copy()
results["cluster"] = cluster

print("Selected K:", BEST_K)
print("Silhouette:", round(silhouette_score(X, cluster), 4))
print("Calinski-Harabasz:", round(calinski_harabasz_score(X, cluster), 2))
print("Davies-Bouldin:", round(davies_bouldin_score(X, cluster), 4))
print("\nCluster sizes:")
display(results["cluster"].value_counts().sort_index().to_frame("countries"))

## 9. PCA visualisation

PCA is used only for a two-dimensional visualisation. K-Means itself was fitted on all nine scaled indicators.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X)

pca_df = pd.DataFrame({
    "PC1": X_pca[:,0],
    "PC2": X_pca[:,1],
    "cluster": cluster,
    "country": data["country"]
})

fig, ax = plt.subplots(figsize=(10, 7))
for c in sorted(pca_df["cluster"].unique()):
    g = pca_df[pca_df["cluster"] == c]
    ax.scatter(g["PC1"], g["PC2"], s=45, alpha=0.75, label=f"Cluster {c}")

ax.set_title("Country development clusters — PCA projection")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print("2D PCA explained variance:", round(pca.explained_variance_ratio_.sum(), 4))

## 10. Cluster profiles in original units

In [ ]:
cluster_profile = (
    results.groupby("cluster")[features]
    .median()
    .round(2)
)
cluster_sizes = results.groupby("cluster").size().rename("countries")
cluster_profile = cluster_profile.join(cluster_sizes)

display(cluster_profile)

profile_z = pd.DataFrame(
    scaler.transform(cluster_profile[features]),
    index=cluster_profile.index,
    columns=features
)

fig, ax = plt.subplots(figsize=(12, max(4, BEST_K * 1.2)))
im = ax.imshow(profile_z, aspect="auto", vmin=-2.5, vmax=2.5)
ax.set_xticks(range(len(features)), features, rotation=45, ha="right")
ax.set_yticks(range(BEST_K), [f"Cluster {i}" for i in cluster_profile.index])
ax.set_title("Cluster profiles relative to the global dataset")
plt.colorbar(im, ax=ax, label="standard deviations from overall mean")
plt.tight_layout()
plt.show()

## 11. Turn anonymous cluster numbers into interpretable development profiles

A post-hoc **development-need score** is created from standardised indicators.

Higher need is associated with:
- higher child mortality
- higher fertility
- higher inflation
- lower life expectancy
- lower income
- lower GDP per capita

This score is **not used to fit K-Means**. It is only used afterwards to interpret and rank the discovered clusters.

In [ ]:
z = pd.DataFrame(X, columns=features, index=data.index)

need_score = (
    + 1.25 * z["child_mort"]
    + 0.70 * z["total_fer"]
    + 0.35 * z["inflation"]
    - 1.10 * z["life_expec"]
    - 1.00 * z["income"]
    - 1.00 * z["gdpp"]
)

results["development_need_score"] = need_score

cluster_need = (
    results.groupby("cluster")["development_need_score"]
    .median()
    .sort_values(ascending=False)
)

ordered_clusters = cluster_need.index.tolist()

if BEST_K == 3:
    semantic_names = {
        ordered_clusters[0]:"Higher development-need profile",
        ordered_clusters[1]:"Transition / middle profile",
        ordered_clusters[2]:"Stronger development profile"
    }
else:
    semantic_names = {
        c:f"Development profile {rank+1} — need rank {rank+1}/{BEST_K}"
        for rank,c in enumerate(ordered_clusters)
    }

results["cluster_profile_name"] = results["cluster"].map(semantic_names)

display(
    results.groupby(["cluster","cluster_profile_name"])
    .agg(
        countries=("country","size"),
        median_need_score=("development_need_score","median"),
        median_child_mort=("child_mort","median"),
        median_income=("income","median"),
        median_life_expectancy=("life_expec","median"),
        median_gdpp=("gdpp","median")
    )
    .round(2)
)

## 12. Aid-priority ranking

Within the highest-need cluster, countries are ranked by the post-hoc development-need score. This makes the clustering actionable for a hypothetical humanitarian resource-allocation discussion.

It is a portfolio decision-support example, **not an official recommendation from HELP International**.

In [ ]:
highest_need_cluster = ordered_clusters[0]

priority = (
    results.loc[results["cluster"] == highest_need_cluster, [
        "country","cluster","cluster_profile_name",
        "development_need_score","child_mort","income",
        "inflation","life_expec","total_fer","gdpp"
    ]]
    .sort_values("development_need_score", ascending=False)
    .reset_index(drop=True)
)

priority.index = priority.index + 1
priority.index.name = "priority_rank"

print("Top 20 countries in the highest-need statistical cluster:")
display(priority.head(20).round(3))

priority.to_csv("project9_country_aid_priority_ranking.csv")
results.to_csv("project9_country_cluster_assignments.csv", index=False)

## 13. Representative countries per cluster

The nearest country to each centroid is a useful plain-English way to describe what each cluster looks like.

In [ ]:
distances = final_kmeans.transform(X)

representatives = []
for c in range(BEST_K):
    idx = np.where(cluster == c)[0]
    local = idx[np.argmin(distances[idx, c])]
    representatives.append({
        "cluster":c,
        "profile":semantic_names[c],
        "representative_country":data.loc[local,"country"],
        "distance_to_centroid":distances[local,c]
    })

representatives = pd.DataFrame(representatives).set_index("cluster")
display(representatives.round(3))

## 14. Stability across random seeds

A useful clustering solution should not radically change when K-Means starts from different centroid initialisations.

Adjusted Rand Index (ARI) compares each repeated run with the reference clustering:
- 1.0 = identical grouping
- 0 ≈ chance-level agreement

In [ ]:
stability_rows = []
for seed in range(20):
    m = KMeans(
        n_clusters=BEST_K,
        init="k-means++",
        n_init=30,
        random_state=seed
    )
    labels = m.fit_predict(X)
    stability_rows.append({
        "seed":seed,
        "ARI_vs_reference":adjusted_rand_score(cluster, labels)
    })

stability = pd.DataFrame(stability_rows)
display(stability.round(4))
print("Mean ARI:", round(stability["ARI_vs_reference"].mean(), 4))
print("Minimum ARI:", round(stability["ARI_vs_reference"].min(), 4))

fig, ax = plt.subplots(figsize=(8,4.5))
ax.plot(stability["seed"], stability["ARI_vs_reference"], marker="o")
ax.set_ylim(-0.05,1.05)
ax.set_title("K-Means stability across random seeds")
ax.set_xlabel("Random seed")
ax.set_ylabel("ARI vs reference")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 15. Reusable assignment function

This can assign a new country observation to one of the learned development profiles, provided the same nine indicators are available.

In [ ]:
def assign_country_profile(new_country_frame):
    missing = [c for c in features if c not in new_country_frame.columns]
    if missing:
        raise ValueError(f"Missing required features: {missing}")

    X_new = scaler.transform(new_country_frame[features])
    labels = final_kmeans.predict(X_new)
    dists = final_kmeans.transform(X_new)

    return pd.DataFrame({
        "cluster":labels,
        "profile":[semantic_names[int(c)] for c in labels],
        "distance_to_centroid":dists[np.arange(len(labels)), labels]
    }, index=new_country_frame.index)

demo = assign_country_profile(data.head(5))
display(pd.concat([data.head(5)[["country"]], demo], axis=1))

## 16. Integrity checks and final result

In [ ]:
assert len(results) == 167
assert results["country"].nunique() == 167
assert len(features) == 9
assert np.isfinite(X).all()
assert results["cluster"].nunique() == BEST_K
assert set(results["cluster"].unique()) == set(range(BEST_K))
assert stability["ARI_vs_reference"].between(-1,1).all()

final_sil = silhouette_score(X, cluster)
final_db = davies_bouldin_score(X, cluster)

print("PROJECT 9 ENHANCED — FINAL RESULT")
print("="*72)
print(f"Countries: {len(results)}")
print(f"Indicators: {len(features)}")
print(f"Elbow K: {elbow_k} ({elbow_method})")
print(f"Selected K by silhouette: {BEST_K}")
print(f"Final silhouette: {final_sil:.4f}")
print(f"Final Davies-Bouldin: {final_db:.4f}")
print(f"Mean seed-stability ARI: {stability['ARI_vs_reference'].mean():.4f}")
print()
print("Cluster profiles:")
for c in ordered_clusters:
    count = int((results["cluster"]==c).sum())
    print(f"- Cluster {c}: {semantic_names[c]} — {count} countries")
print()
print("Highest-need cluster:", highest_need_cluster)
print("Top five statistical priority countries:")
print(priority["country"].head(5).tolist())
print()
print("All integrity checks passed ✅")

## Interview-ready explanation

> “My university lab used K-Means and the K-Elbow Visualizer to cluster 167 countries into development groups. I preserved that work, then built a separate enhanced version. The key correction was standardising all nine indicators before distance-based clustering so GDP and income did not dominate. I then selected K using multiple internal metrics, visualised the structure with PCA, profiled each cluster in original units, tested stability across random seeds, and added a transparent post-hoc development-need score for a humanitarian resource-allocation use case.”

### Limitations
- the dataset is a historical static snapshot, not a live economic feed;
- cluster membership is sensitive to feature definitions and scaling;
- the aid-priority score is an illustrative analytical rule, not a policy recommendation;
- country-level averages hide regional and within-country inequality;
- development should never be reduced to a single cluster label for real policy decisions.